# Задание 6.2. Сигнал на входе и выходе канала
**Зелёный и жёлтый уровни**

Рассмотрим перенос примеси вдоль канала длины 1:
$$u_t+cu_x=0,\qquad x\in[0,1],\quad c=1,\quad u(x,0)=0.$$
На входе задаётся концентрация $u(0,t)=q(t)$, где
$$
q(t)=\begin{cases}
\sin^2(\pi t/\tau),& 0\le t\le \tau,\\
0,&\text{иначе},
\end{cases}
\qquad \tau=0.2.
$$

В зелёной части выполним расчёт до $t=1.5$ на четырёх сетках с числом Куранта $C=0.8$. Сравним профили и выходные сигналы с точным решением, проверим баланс примеси и оценим, как меняются высота, длительность и время прихода импульса.



В жёлтой части (разделы 7–10) скорость меняется вдоль канала. Сравним перенос плотности и перенос значения вдоль характеристики.

## 1. Точное решение

При $c>0$ характеристики направлены слева направо. Сигнал, наблюдаемый в точке $x$ в момент $t$, поступил на вход в момент $t-x/c$. Если это время отрицательно, сигнал ещё не дошёл и канал в этой точке остаётся чистым. Таким образом,
$$u(x,t)=q(t-x/c),\qquad u(1,t)=q(t-1).$$

Точный выход — копия входного импульса, задержанная на единицу времени. Она ненулевая между $1$ и $1.2$, её максимум равен $1$ при $t=1.1$.

В численной схеме хранятся **средние по ячейкам**. Для вычисления ошибки будем сравнивать их с точными средними, а не со значениями в центрах. Обозначим $Q(s)=\int_{-\infty}^{s}q(r)\,dr$. Тогда точное среднее по ячейке $[a,b]$:
$$
\bar u_{[a,b]}(t)=\frac{c}{b-a}
\left[Q(t-a/c)-Q(t-b/c)\right].
$$
Первообразная вычисляется аналитически: если $r=\min(\tau,\max(0,s))$, то
$$Q(s)=\frac{r}{2}-\frac{\tau}{4\pi}\sin\frac{2\pi r}{\tau}.$$
За весь импульс в канал входит $cQ(\tau)=0.1$ единицы количества примеси.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

c, tau, t_end, C = 1.0, 0.2, 1.5, 0.8
counts = [80, 160, 320, 640]
colors = ["#b86743", "#28728a", "#458353", "#644c91"]
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.18,
    "animation.frame_format": "jpeg", "animation.embed_limit": 30,
})


def q(t):
    t = np.asarray(t)
    return np.where((t >= 0) & (t <= tau), np.sin(np.pi*t/tau)**2, 0.0)


def primitive(t):
    r = np.clip(t, 0, tau)
    return r/2 - tau/(4*np.pi)*np.sin(2*np.pi*r/tau)


def exact_average(faces, t):
    return -c*np.diff(primitive(t-faces/c))/np.diff(faces)

## 2. Схема upwind и дискретный баланс

Делим канал на $N$ ячеек длины $h=1/N$. Обозначим через $U_j^n$ среднее в ячейке $j$ при $t_n=n\Delta t$, а через $F_j^n$ — поток через грань $x=jh$. Ячейки нумеруются от $0$ до $N-1$, грани — от $0$ до $N$.

Поскольку $c>0$, на внутренней грани поток берёт концентрацию из ячейки слева. Вход и выход задаются так:
$$
F_0^n=cq(t_n),\qquad
F_j^n=cU_{j-1}^n,\quad 1\le j\le N.
$$
В частности, $F_N^n=cU_{N-1}^n$ — поток из последней ячейки. **Дополнительного условия на выходе нет.** Обновление ячейки:
$$
U_j^{n+1}=U_j^n-\frac{\Delta t}{h}(F_{j+1}^n-F_j^n).
$$
Для внутренних ячеек это взвешенное среднее $(1-C)U_j^n+C U_{j-1}^n$. При $0\le C\le1$ веса неотрицательны. Во всех опытах берём $\Delta t=0.8h/c$; выбранные шаги делят $t=1.5$ без остатка.

Если просуммировать обновление по всем ячейкам, внутренние потоки сократятся. Останется
$$
M^{n+1}-M^n=\Delta t(F_0^n-F_N^n),\qquad
M^n=h\sum_{j=0}^{N-1}U_j^n.
$$
Накопленные приток $I^n$ и отток $O^n$ считаются по тем же потокам и тем же шагам, что и обновление $U$. Невязка:
$$R^n=M^n-M^0-I^n+O^n.$$
В точной арифметике она равна нулю. Этот баланс проверяет количество примеси, а форму профиля проверяем отдельно.

In [ ]:
def solve_channel(N):
    h = 1.0/N
    dt = C*h/c
    steps = int(round(t_end/dt))
    if not np.isclose(steps*dt, t_end, atol=1e-12, rtol=0):
        raise ValueError("Выберите сетку, на которой t_end кратно Δt.")

    t = dt*np.arange(steps+1)
    faces = np.linspace(0.0, 1.0, N+1)
    u = np.zeros((steps+1, N))
    flux = np.empty(N+1)
    inlet = q(t)
    fin, fout = np.empty(steps), np.empty(steps)

    for n in range(steps):
        flux[0] = c*inlet[n]
        flux[1:] = c*u[n]
        fin[n], fout[n] = flux[0], flux[-1]
        u[n+1] = u[n] - (dt/h)*np.diff(flux)

    mass = h*u.sum(axis=1)
    incoming = np.r_[0.0, np.cumsum(dt*fin)]
    outgoing = np.r_[0.0, np.cumsum(dt*fout)]
    residual = mass-mass[0]-incoming+outgoing

    # Точки на выходе относятся к началу шага t_n.
    # Последнее значение нужно только для графика при t_end.
    outlet = np.r_[fout/c, u[-1, -1]]
    return dict(h=h, dt=dt, t=t, faces=faces, u=u, outlet=outlet,
                mass=mass, incoming=incoming, outgoing=outgoing,
                residual=residual)


runs = {N: solve_channel(N) for N in counts}
print(f"{'N':>6} {'h':>12} {'Δt':>12} {'C':>8} {'Шагов':>8}")
for N, r in runs.items():
    print(f"{N:6d} {r['h']:12.7f} {r['dt']:12.7f} "
          f"{c*r['dt']/r['h']:8.3f} {len(r['t'])-1:8d}")

## 3. Профиль импульса внутри канала

Сравниваем расчёты в одинаковые моменты физического времени. Точный профиль — непрерывная кривая, численный — ступеньки со значениями средних в ячейках. К $t=1.1$ часть импульса уже вышла, а к $t=1.24$ точный профиль равен нулю во всём канале.

Для численного сравнения при $t=0.6$ измеряем среднюю интегральную ошибку по каналу:
$$E_x=h\sum_j |U_j-\bar u_j^{\mathrm{exact}}|.$$
Здесь $\bar u_j^{\mathrm{exact}}$ — аналитические средние из раздела 1. Длина канала равна единице, поэтому $E_x$ также равна средней абсолютной ошибке по пространству.

In [ ]:
def time_index(r, t):
    n = int(round(t/r["dt"]))
    if not np.isclose(r["t"][n], t, atol=1e-12, rtol=0):
        raise ValueError("Момент сравнения не попал на временную сетку.")
    return n


sample_times = [0.30, 0.60, 1.10, 1.24]
xx = np.linspace(0, 1, 1001)
fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.2), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, t in zip(axes.flat, sample_times):
    for N, color in zip(counts, colors):
        r = runs[N]
        ax.stairs(r["u"][time_index(r, t)], r["faces"], baseline=None,
                  lw=1.1, color=color, label=f"N = {N}")
    ax.plot(xx, q(t-xx/c), "k--", lw=1.45, label="Точное")
    ax.set(title=f"t = {t:.2f}", xlim=(0, 1), ylim=(-0.04, 1.08),
           xlabel="x", ylabel="Концентрация")
axes[0, 0].legend(fontsize=8, ncol=2, loc="upper right")
fig.suptitle("Перенос импульса: сравнение на общих моментах времени")
plt.show()

profile_error = {}
for N, r in runs.items():
    t = 0.6
    j = time_index(r, t)
    profile_error[N] = r["h"]*np.abs(r["u"][j]-exact_average(r["faces"], t)).sum()

## 4. Входной и выходной сигналы

Выходная концентрация в численном расчёте — $F_N^n/c=U_{N-1}^n$ при $t=t_n$. Сопоставим её с $q(t-1)$ и отдельно покажем исходный сигнал $q(t)$.

Измеряем четыре характеристики:

- **Высота** — максимальное значение на временной сетке.
- **Момент пика** — время этого узлового максимума; его точность ограничена шагом времени.
- **Длительность** — ширина на половине *собственного* максимума (FWHM).
- **Момент прихода** — первое пересечение общего порога $0.05$ снизу вверх. Порог позволяет не принимать за приход практически нулевой хвост размытого решения.

Время пересечения уровня уточняем линейно между соседними отсчётами. Для точного выхода ширина на полувысоте равна $0.1$, а момент прихода по порогу $0.05$ равен $1+\frac{\tau}{\pi}\arcsin\sqrt{0.05}$. Этот момент отличается от начала точного ненулевого сигнала ($t=1$), потому что после $t=1$ ещё нужно набрать высоту $0.05$.

In [ ]:
def crossing(t, signal, level, j):
    # Время пересечения между узлами j и j+1.
    fraction = (level-signal[j])/(signal[j+1]-signal[j])
    return t[j]+fraction*(t[j+1]-t[j])


def signal_metrics(t, signal, threshold=0.05):
    peak_index = int(np.argmax(signal))
    peak = signal[peak_index]
    half = np.flatnonzero(signal >= peak/2)
    first = np.flatnonzero((signal[:-1] < threshold) & (signal[1:] >= threshold))[0]
    left = crossing(t, signal, peak/2, half[0]-1)
    right = crossing(t, signal, peak/2, half[-1])
    return peak, t[peak_index], right-left, crossing(t, signal, threshold, first)


threshold = 0.05
exact_arrival = 1/c+tau/np.pi*np.arcsin(np.sqrt(threshold))
metrics = {N: signal_metrics(r["t"], r["outlet"], threshold) for N, r in runs.items()}
print(f"{'Сетка':>8} {'Высота':>11} {'Время пика':>13} {'Ширина':>11} {'Приход':>11}")
for N, values in metrics.items():
    print(f"{N:8d} {values[0]:11.6f} {values[1]:13.6f} "
          f"{values[2]:11.6f} {values[3]:11.6f}")
print(f"{'Точно':>8} {1.0:11.6f} {1/c+tau/2:13.6f} "
      f"{tau/2:11.6f} {exact_arrival:11.6f}")

t_dense = np.linspace(0, t_end, 1501)
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.3),
                         gridspec_kw={"width_ratios": [1.45, 1]},
                         constrained_layout=True)
axes[0].plot(t_dense, q(t_dense), color="0.55", lw=1.6, label="Вход")
for ax in axes:
    for (N, r), color in zip(runs.items(), colors):
        ax.plot(r["t"], r["outlet"], color=color, lw=1.35, label=f"Выход, N = {N}")
    ax.plot(t_dense, q(t_dense-1/c), "k--", lw=1.45, label="Точный выход")
    ax.set(xlabel="t", ylabel="Концентрация")
axes[0].set(title="Сигналы на входе и выходе", xlim=(0, 1.5), ylim=(-0.02, 1.06))
axes[0].legend(fontsize=8, loc="upper center", ncol=2)
axes[1].axhline(threshold, color="0.55", ls=":", lw=1)
axes[1].scatter([values[3] for values in metrics.values()], [threshold]*len(counts),
                c=colors, s=22, zorder=5)
axes[1].scatter(exact_arrival, threshold, color="black", s=22, zorder=5)
axes[1].set(title="Передний край: пересечение уровня 0.05",
            xlim=(0.935, 1.06), ylim=(-0.006, 0.4))
plt.show()

На грубой сетке $N=80$ пик падает с $1$ до $0.641$, а ширина на полувысоте увеличивается с $0.100$ до $0.147$. При сгущении сетки выход становится выше и уже: при $N=640$ высота $0.929$, ширина $0.105$.

Передний край размывается: уровень $0.05$ достигается раньше точного момента $1.01436$. Например, при $N=80$ это происходит в $0.96359$, а при $N=640$ — в $1.00414$. Раннее достижение малого порога и положение максимума — разные характеристики. На всех четырёх сетках наибольший отсчёт сигнала находится при $t=1.10$; заметного запаздывания узлового пика нет.

Обновление upwind смешивает соседние значения, поэтому оно сглаживает импульс. Ведущий коэффициент численной диффузии (см. конспект занятия) равен $D_{\mathrm{num}}=ch(1-C)/2$. При фиксированном $C$ он уменьшается вместе с $h$, что объясняет ослабление размытия на мелких сетках.

In [ ]:
def trapezoid_integral(values, t):
    # Площадь каждой трапеции: среднее соседних высот × длина шага.
    return np.sum(0.5*(values[:-1]+values[1:])*np.diff(t))


outlet_error = {
    N: trapezoid_integral(np.abs(r["outlet"]-q(r["t"]-1/c)), r["t"])
    for N, r in runs.items()
}
print("Ошибки: E_x — профиль при t=0.6, E_t — интеграл ошибки выхода по времени.")
print(f"{'N':>6} {'E_x':>13} {'E_t':>13}")
for N in counts:
    print(f"{N:6d} {profile_error[N]:13.6e} {outlet_error[N]:13.6e}")

h_values = np.array([runs[N]["h"] for N in counts])
fig, ax = plt.subplots(figsize=(7.5, 4.1), constrained_layout=True)
ax.loglog(h_values, [profile_error[N] for N in counts], "o-", color=colors[1],
          label="Ошибка профиля при t = 0.6")
ax.loglog(h_values, [outlet_error[N] for N in counts], "s-", color=colors[0],
          label="Ошибка выходного сигнала")
ax.loglog(h_values, profile_error[640]*h_values/h_values[-1], "k:",
          label="Прямая наклона 1")
ax.set(xlabel="Размер ячейки h", ylabel="Интегральная абсолютная ошибка",
       title="Точность при сгущении сетки, C = 0.8")
ax.legend(fontsize=9)
plt.show()

## 5. Баланс количества примеси

На первом графике показаны $M^n$, $I^n$ и $O^n$ для сетки $N=160$. Пока примесь поступает, её количество в канале растёт. После окончания подачи и до прихода к выходу оно почти постоянно; затем уменьшается, а накопленный отток растёт. К концу расчёта практически вся примесь вышла.

На втором графике — невязка $R^n$ для всех сеток. Значения порядка $10^{-16}$ соответствуют округлению. Использование аналитического выходного сигнала вместо $F_N^n$ в накопленном оттоке нарушило бы эту проверку, поскольку это был бы другой поток.

In [ ]:
demo = runs[160]
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.1), constrained_layout=True)
axes[0].plot(demo["t"], demo["mass"], lw=2, color=colors[1], label="В канале M")
axes[0].plot(demo["t"], demo["incoming"], "--", lw=1.5, color="0.45",
             label="Накопленный приток I")
axes[0].plot(demo["t"], demo["outgoing"], lw=1.7, color=colors[0],
             label="Накопленный отток O")
axes[0].set(title="Количество примеси, N = 160", ylabel="Количество")
axes[0].legend(fontsize=9)
for (N, r), color in zip(runs.items(), colors):
    axes[1].plot(r["t"], 1e16*r["residual"], lw=0.9, color=color, label=f"N = {N}")
axes[1].set(title="Невязка дискретного баланса", ylabel=r"$R^n / 10^{-16}$")
axes[1].legend(fontsize=8, ncol=2)
for ax in axes:
    ax.set(xlabel="t", xlim=(0, 1.5))
plt.show()

print(f"{'N':>6} {'Приток к 1.5':>15} {'Отток к 1.5':>15} "
      f"{'Остаток к 1.5':>15} {'max |R|':>13}")
for N, r in runs.items():
    print(f"{N:6d} {r['incoming'][-1]:15.10f} {r['outgoing'][-1]:15.10f} "
          f"{r['mass'][-1]:15.3e} {np.max(np.abs(r['residual'])):13.3e}")

## 6. Анимация прохождения импульса

Слева — профиль в канале ($N=160$) и точное решение. Справа — выходные сигналы; вертикальная линия указывает то же время, что и профиль слева. В кадры попадает каждый третий шаг интегрирования, но расчёт выполняется на всех шагах. Нажмите ▶ для воспроизведения.

In [ ]:
frame_nodes = np.arange(0, len(demo["t"]), 3)
fig, (ax_u, ax_out) = plt.subplots(1, 2, figsize=(10.4, 4.0))
fig.subplots_adjust(left=0.075, right=0.98, bottom=0.15, top=0.85, wspace=0.27)

profile = ax_u.stairs(demo["u"][0], demo["faces"], baseline=None,
                      color=colors[1], lw=1.8, label="Upwind")
exact_line, = ax_u.plot(xx, q(-xx/c), "k--", lw=1.4, label="Точное")
ax_u.set(xlabel="x", ylabel="Концентрация", xlim=(0, 1), ylim=(-0.03, 1.06),
         title="Профиль в канале")
ax_u.legend(loc="upper right", fontsize=8)

ax_out.plot(demo["t"], demo["outlet"], color=colors[1], lw=1.6, label="Upwind")
ax_out.plot(t_dense, q(t_dense-1/c), "k--", lw=1.4, label="Точный выход")
cursor = ax_out.axvline(0, color=colors[0], lw=1.3)
point, = ax_out.plot([0], [0], "o", color=colors[1], ms=5)
ax_out.set(xlabel="t", ylabel="Концентрация", xlim=(0, 1.5), ylim=(-0.03, 1.06),
           title="Сигнал на выходе")
ax_out.legend(loc="upper left", fontsize=8)
title = fig.suptitle("")


def update(n):
    t = demo["t"][n]
    profile.set_data(values=demo["u"][n])
    exact_line.set_ydata(q(t-xx/c))
    cursor.set_xdata([t, t])
    point.set_data([t], [demo["outlet"][n]])
    title.set_text(f"Прохождение импульса · t = {t:.3f}")
    return profile, exact_line, cursor, point, title


movie = FuncAnimation(fig, update, frames=frame_nodes, interval=80, blit=False)
with plt.rc_context({"savefig.dpi": 75}):
    animation = movie.to_jshtml(fps=12, default_mode="once")
plt.close(fig)
display(HTML(animation))

## Итог зелёного уровня

1. При $c=1$ точный выход повторяет вход с задержкой 1. Схема upwind даёт более низкий и широкий импульс: на сетке $N=80$ пик ниже точного примерно на 36%, а ширина на полувысоте больше примерно на 47%.
2. Слабая передняя часть численного сигнала приходит раньше: фиксированный уровень $0.05$ пересекается раньше точного. При этом узловой максимум в этих опытах приходится на $t=1.10$. Ранний фронт не означает такого же сдвига всего импульса.
3. При увеличении $N$ от 80 до 640 при одном и том же $C=0.8$ уменьшаются ошибки профиля и выходного сигнала, восстанавливается высота и уменьшается ширина.
4. Баланс притока, оттока и количества примеси внутри канала выполняется с точностью округления на всех сетках. Сохранение количества примеси совместимо с заметным искажением её распределения.

## 7. Жёлтый уровень: переменная скорость и точные решения

Теперь $c(x)=1+0.5\sin(2\pi x)$, поэтому $0.5\le c\le1.5$. Начальные данные нулевые. Входной поток $c(0)u(0,t)=q(t)$ задаётся тем же импульсом, что в зелёной части. Поскольку $c(0)=1$, в обоих расчётах входная плотность равна $q(t)$. Выход свободный.

Сравниваем два уравнения:
$$
\begin{aligned}
&u_t+(cu)_x=0 &&\text{(перенос плотности)},\\
&u_t+cu_x=0 &&\text{(перенос значения)}.
\end{aligned}
$$
Характеристики обоих уравнений движутся по $dx/dt=c(x)$. Частица в точке $x$ вышла со входа в момент $t-s(x)$, где
$$s(x)=\int_0^x \frac{d\xi}{c(\xi)}.$$
Для второго уравнения вдоль характеристики сохраняется $u$. Для первого раскрытие производной даёт $du/dt=-c'(x)u$, откуда $d(cu)/dt=0$. Поэтому точные решения:
$$
u_{\rm cons}(x,t)=\frac{q(t-s(x))}{c(x)},\qquad
u_{\rm adv}(x,t)=\frac{q(t-s(x))}{c(0)}=q(t-s(x)).
$$

Используем аналитическое выражение для $s$. Для $\theta=\pi x$ и $\kappa=\sqrt{1-0.5^2}$ положим
$$X=\cos\theta+0.5\sin\theta,\qquad Y=\kappa\sin\theta.$$
Угол $\operatorname{atan2}(Y,X)$ меняется от $0$ до $\pi$. Его производная по $\theta$ равна $\kappa/(X^2+Y^2)=\kappa/c(x)$. Значит,
$$s(x)=\frac{\operatorname{atan2}(Y,X)}{\pi\kappa}.$$
Для независимого контроля сравним формулу с составными трапециями на двух мелких сетках. Точный выходной поток обеих моделей совпадает с $q(t-s(1))$, так как $c(0)=c(1)=1$. Последняя часть точного импульса выходит при $t=s(1)+0.2$.

In [ ]:
amplitude = 0.5
kappa = np.sqrt(1-amplitude**2)


def channel_speed(x):
    return 1 + amplitude*np.sin(2*np.pi*x)


def travel_time(x):
    angle = np.pi*np.asarray(x)
    sn, cs = np.sin(angle), np.cos(angle)
    return np.arctan2(kappa*sn, cs+amplitude*sn)/(np.pi*kappa)


def yellow_exact(x, t, mode):
    pulse = q(t-travel_time(x))
    return pulse/channel_speed(x) if mode == "cons" else pulse


def travel_check(n):
    x = np.linspace(0, 1, n+1)
    inverse = 1/channel_speed(x)
    numeric = np.r_[0, np.cumsum((inverse[:-1]+inverse[1:])*np.diff(x)/2)]
    return np.max(np.abs(numeric-travel_time(x)))


passage = float(travel_time(1.0))
print(f"Время прохождения s(1) = {passage:.12f}")
print(f"Конец точного выходного импульса: {passage+tau:.12f}")
for n in [16384, 32768]:
    print(f"Трапеции, N = {n:5d}: max |s_числ − s| = {travel_check(n):.3e}")

## 8. Две схемы upwind и сравнение профилей

Используем одни и те же равномерные ячейки и время для обеих моделей. $c_j$ — скорость в центре ячейки, $c_{j\pm1/2}$ — скорости на её гранях. Шаг $\Delta t=0.8h/1.5$, поэтому $\max c\,\Delta t/h=0.8$. Расчёт доводим до $t=2$, чтобы успел выйти и численно размытый хвост.

**Плотность:** храним средние $U_j$ по ячейкам. Для положительной скорости на правой грани берём $F_{j+1/2}=c_{j+1/2}U_j$. Входной поток — $q(t_n)$. Обновляем разностью потоков:
$$U_j^{n+1}=U_j^n-\frac{\Delta t}{h}(F_{j+1/2}^n-F_{j-1/2}^n).$$

**Перенос значения:** применяем левую разность к $u_x$ в центре:
$$U_j^{n+1}=U_j^n-\frac{\Delta t}{h}c_j(U_j^n-U_{j-1}^n).$$
Для первой ячейки слева берём входное значение $q(t_n)/c(0)$. Для обеих моделей выходной поток оцениваем через концентрацию в последней ячейке и скорость на выходной грани. Вторая модель использует эти потоки для диагностики, её обновление записано через левую разность.

Для количественной проверки консервативной схемы используем точные средние по ячейкам: после замены $ds=dx/c(x)$ среднее на $[a,b]$ равно
$$
\frac{Q(t-s(a))-Q(t-s(b))}{h},
\qquad Q(s)=\int_{-\infty}^{s}q(r)\,dr .
$$
Во второй схеме сравниваем с точным значением в центре ячейки.

In [ ]:
yellow_end, yellow_cfl = 2.0, 0.8
yellow_counts = [80, 160, 320, 640]
names = {"cons": "Консервативная", "adv": "Неконсервативная"}
model_colors = {"cons": "#28728a", "adv": "#a44865"}


def solve_variable(n, mode):
    faces = np.linspace(0, 1, n+1)
    centers = (faces[:-1]+faces[1:])/2
    cf, cc = channel_speed(faces), channel_speed(centers)
    h = 1/n
    dt = yellow_cfl*h/(1+amplitude)
    steps = int(round(yellow_end/dt))
    if not np.isclose(steps*dt, yellow_end, rtol=0, atol=1e-12):
        raise ValueError("На выбранной сетке конечное время должно быть кратно шагу.")
    t = dt*np.arange(steps+1)
    inlet = q(t)
    u = np.zeros((steps+1, n))
    fin, fout, source_rate = np.empty(steps), np.empty(steps), np.zeros(steps)

    for k in range(steps):
        current = u[k]
        left = np.r_[inlet[k]/cf[0], current[:-1]]
        flux = np.r_[inlet[k], cf[1:]*current]
        fin[k], fout[k] = flux[0], flux[-1]
        if mode == "cons":
            u[k+1] = current - dt/h*np.diff(flux)
        elif mode == "adv":
            u[k+1] = current - dt/h*cc*(current-left)
            source_rate[k] = np.sum((cf[1:]-cc)*current+(cc-cf[:-1])*left)
        else:
            raise ValueError("Неизвестная модель.")

    incoming = np.r_[0, np.cumsum(dt*fin)]
    outgoing = np.r_[0, np.cumsum(dt*fout)]
    mass = h*u.sum(axis=1)
    source_total = np.r_[0, np.cumsum(dt*source_rate)]
    residual = mass-mass[0]-incoming+outgoing
    return dict(t=t, dt=dt, h=h, faces=faces, x=centers, u=u, mass=mass,
                incoming=incoming, outgoing=outgoing, residual=residual,
                source_total=source_total, outlet=np.r_[fout, cf[-1]*u[-1, -1]])


yellow_runs = {n: {mode: solve_variable(n, mode) for mode in names}
               for n in yellow_counts}
print(f"{'N':>6} {'Δt':>14} {'Число шагов':>14} {'max c Δt/h':>13}")
for n, pair in yellow_runs.items():
    r = pair["cons"]
    print(f"{n:6d} {r['dt']:14.8f} {len(r['t'])-1:14d} "
          f"{(1+amplitude)*r['dt']/r['h']:13.3f}")

In [ ]:
yellow_demo = yellow_runs[160]
field_x = np.linspace(0, 1, 1001)
profile_times = [0.30, 0.60, 0.88, 1.18]
fig = plt.figure(figsize=(10.8, 7.7), constrained_layout=True)
grid = fig.add_gridspec(3, 2, height_ratios=[0.8, 1.3, 1.3])
ax_speed = fig.add_subplot(grid[0, :])
ax_speed.plot(field_x, channel_speed(field_x), color="0.25", lw=1.8)
ax_speed.axhline(1, color="0.55", ls=":", lw=1)
ax_speed.axvspan(0.5, 1, color="#f0e0c4", alpha=0.55)
ax_speed.set(xlabel="x", ylabel="c(x)", xlim=(0, 1), ylim=(0.38, 1.62),
             title="Скорость потока: фоном выделена медленная половина канала")
for i, t in enumerate(profile_times):
    ax = fig.add_subplot(grid[1+i//2, i%2], sharex=ax_speed)
    ax.axvspan(0.5, 1, color="#f0e0c4", alpha=0.55)
    for mode, r in yellow_demo.items():
        k = time_index(r, t)
        ax.plot(field_x, yellow_exact(field_x, t, mode), "--", lw=1.45,
                color=model_colors[mode], label=f"{names[mode]}: точно")
        ax.plot(r["x"], r["u"][k], lw=1.6, color=model_colors[mode],
                label=f"{names[mode]}: upwind")
    ax.set(title=f"t = {t:.2f}", xlabel="x", ylabel="u", ylim=(-0.05, 2.10))
    if i == 0:
        ax.legend(fontsize=7.4, loc="upper right")
fig.suptitle("Скорость и профили обеих моделей в одинаковые моменты времени")
plt.show()

Если две порции поступили на вход с небольшой разницей по времени $\delta t$, то около точки $x$ они разделены расстоянием примерно $c(x)\delta t$. В медленной области это расстояние меньше. В консервативной модели то же количество примеси занимает меньший отрезок, поэтому плотность увеличивается. Например, в точке $x=0.75$ скорость равна $0.5$, и при прохождении вершины импульса точная плотность равна $1/0.5=2$. В модели $u_t+cu_x=0$ вершина сохраняет значение 1. Схема upwind дополнительно сглаживает оба численных профиля.

## 9. Выходные сигналы и баланс

Проверяем точность при фиксированном числе Куранта на четырёх сетках. $E_x$ — интегральная абсолютная ошибка профиля при $t=0.88$, $E_{\rm out}$ — интеграл абсолютной ошибки выходного потока по времени от 0 до 2. Точные выходы совпадают, поэтому сравнение только у выхода не различает эти модели.

In [ ]:
profile_t = 0.88
yellow_errors = {}
print(f"{'N':>6} {'Модель':>19} {'E_x':>14} {'E_out':>14}")
for n, pair in yellow_runs.items():
    for mode, r in pair.items():
        if mode == "cons":
            reference = -np.diff(primitive(profile_t-travel_time(r["faces"])))/r["h"]
        else:
            reference = yellow_exact(r["x"], profile_t, mode)
        ex_profile = r["h"]*np.sum(np.abs(r["u"][time_index(r, profile_t)]-reference))
        ex_outlet = trapezoid_integral(np.abs(r["outlet"]-q(r["t"]-passage)), r["t"])
        yellow_errors[n, mode] = ex_profile, ex_outlet
        print(f"{n:6d} {names[mode]:>19} {ex_profile:14.6e} {ex_outlet:14.6e}")

fig, axes = plt.subplots(1, 2, figsize=(11.4, 4.3), constrained_layout=True)
dense_t = np.linspace(0, yellow_end, 1201)
axes[0].plot(dense_t, q(dense_t-passage), "k--", lw=1.4, label="Точный выход")
for n, alpha in [(80, 0.65), (640, 1.0)]:
    for mode, style in [("cons", "-"), ("adv", ":")]:
        r = yellow_runs[n][mode]
        axes[0].plot(r["t"], r["outlet"], style, color=model_colors[mode], alpha=alpha,
                     lw=1.8, label=f"{names[mode]}, N={n}")
axes[0].set(xlabel="t", ylabel="Выходной поток", xlim=(0.8, 1.65),
            title="Выход: обе модели и точное решение")
axes[0].legend(fontsize=7.4)
hs = 1.0/np.array(yellow_counts)
for mode in names:
    err = np.array([yellow_errors[n, mode] for n in yellow_counts])
    axes[1].loglog(hs, err[:, 0], "o-", color=model_colors[mode],
                   label=f"Профиль: {names[mode].lower()}")
    axes[1].loglog(hs, err[:, 1], "s--", color=model_colors[mode],
                   label=f"Выход: {names[mode].lower()}")
axes[1].set(xlabel="Размер ячейки h", ylabel="Интегральная ошибка",
            title="Сгущение сетки при max c Δt/h = 0.8")
axes[1].legend(fontsize=7.4)
plt.show()

На выбранных сетках численные выходные кривые двух моделей практически накладываются. При этом их профили внутри канала заметно различаются, как видно в разделе 8.

В обоих расчётах считаем физическое количество примеси $M=h\sum_j U_j$, накопленный приток $I$ и отток $O$. Проверяем $R=M-M^0-I+O$.

- В консервативной схеме внутренние потоки при суммировании сокращаются, и $R$ должен быть нулевым с точностью округления.
- Неконсервативное уравнение можно переписать как $u_t+(cu)_x=c'(x)u$. Поэтому для него $R$ отражает накопленное изменение количества **внутри** канала, связанное с $c'(x)u$.

В коде для второй схемы считаем также дискретный аналог этого внутреннего вклада. Для каждого шага переписываем левую разность как разность тех же диагностических потоков плюс источник. Вклад ячейки в интеграл источника равен
$$hS_j=(c_{j+1/2}-c_j)U_j+(c_j-c_{j-1/2})U_{j-1}.$$
Суммируем его по ячейкам и по времени. После вычитания этого вклада остаток должен быть порядка округления. Это проверяет, что ненулевой $R$ второй модели объясняется самим уравнением.

In [ ]:
exact_t = np.linspace(0, yellow_end, 501)
mass_x = np.linspace(0, 1, 4097)
exact_net = primitive(exact_t)-primitive(exact_t-passage)
exact_adv_mass = np.array([trapezoid_integral(yellow_exact(mass_x, t, "adv"), mass_x)
                           for t in exact_t])
exact_adv_residual = exact_adv_mass-exact_net

fig, axes = plt.subplots(2, 2, figsize=(11.2, 7.2), constrained_layout=True)
for i, (mode, r) in enumerate(yellow_demo.items()):
    ax = axes[0, i]
    ax.plot(r["t"], r["mass"], lw=1.8, color=model_colors[mode], label="В канале M")
    ax.plot(r["t"], r["incoming"], "--", color="0.5", label="Приток I")
    ax.plot(r["t"], r["outgoing"], color="#a46c32", label="Отток O")
    ax.plot(exact_t, exact_net if mode == "cons" else exact_adv_mass,
            "k:", lw=1.2, label="Точное M")
    ax.set(title=f"{names[mode]}: N = 160", ylabel="Количество примеси")
    ax.legend(fontsize=8)
for n, color in zip(yellow_counts, colors):
    for i, mode in enumerate(names):
        r = yellow_runs[n][mode]
        scale = 1e16 if mode == "cons" else 1
        axes[1, i].plot(r["t"], scale*r["residual"], color=color, lw=1.0, label=f"N = {n}")
axes[1, 0].set(title="Консервативная: округление", ylabel=r"$R/10^{-16}$")
axes[1, 1].plot(exact_t, exact_adv_residual, "k--", lw=1.4, label="Точное R")
axes[1, 1].set(title="Неконсервативная: внутренний вклад", ylabel="R = M − I + O")
for ax in axes[1]:
    ax.legend(fontsize=7.4, ncol=2)
for ax in axes.flat:
    ax.set(xlabel="t", xlim=(0, yellow_end))
plt.show()

print(f"{'N':>6} {'Модель':>19} {'max |R|':>13} {'max |R − ΣΔt·hS|':>19} {'M(2)':>13}")
for n, pair in yellow_runs.items():
    for mode, r in pair.items():
        print(f"{n:6d} {names[mode]:>19} {np.max(np.abs(r['residual'])):13.3e} "
              f"{np.max(np.abs(r['residual']-r['source_total'])):19.3e} {r['mass'][-1]:13.3e}")

При сгущении уменьшаются ошибки профилей и выхода. В консервативной схеме невязка баланса остаётся порядка $10^{-16}$. Во второй схеме максимум $|R|$ остаётся порядка $0.05$: в середине процесса меняется количество примеси. При учёте дискретного внутреннего вклада остаток снова близок к округлению. К $t=2$ даже на грубой сетке в канале остаётся меньше $10^{-8}$ единицы примеси из поданных $0.1$. Почти нулевой остаток после выхода импульса сам по себе ничего не говорит о балансе в середине расчёта.

## 10. Анимация при переменной скорости

Слева — перенос плотности, справа — перенос значения. Одинаковые масштабы позволяют увидеть усиление в медленной части канала. Пунктир — точное решение, сплошная линия — upwind при $N=160$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), sharex=True, sharey=True)
fig.subplots_adjust(left=0.075, right=0.98, bottom=0.16, top=0.83, wspace=0.16)
numeric_lines, reference_lines = {}, {}
for ax, (mode, r) in zip(axes, yellow_demo.items()):
    ax.axvspan(0.5, 1, color="#f0e0c4", alpha=0.55)
    numeric_lines[mode], = ax.plot(r["x"], r["u"][0], lw=2,
                                   color=model_colors[mode], label="Upwind")
    reference_lines[mode], = ax.plot(field_x, np.zeros_like(field_x),
                                     "k--", lw=1.4, label="Точное")
    ax.set(title=names[mode], xlabel="x", xlim=(0, 1), ylim=(-0.05, 2.10))
    ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("Плотность или переносимое значение")
yellow_title = fig.suptitle("")
yellow_frames = np.arange(0, len(yellow_demo["cons"]["t"]), 6)


def update_yellow(k):
    t = yellow_demo["cons"]["t"][k]
    for mode, r in yellow_demo.items():
        numeric_lines[mode].set_ydata(r["u"][k])
        reference_lines[mode].set_ydata(yellow_exact(field_x, t, mode))
    yellow_title.set_text(f"Перенос с переменной скоростью · t = {t:.3f}")
    return (*numeric_lines.values(), *reference_lines.values(), yellow_title)


yellow_movie = FuncAnimation(fig, update_yellow, frames=yellow_frames, interval=80)
with plt.rc_context({"savefig.dpi": 75}):
    yellow_html = yellow_movie.to_jshtml(fps=12, default_mode="once")
plt.close(fig)
display(HTML(yellow_html))

## Итог жёлтого уровня

При заданном входном потоке консервативное уравнение переносит количество примеси. В медленной части канала уменьшается расстояние между порциями вещества, и плотность растёт как $1/c(x)$ при одном и том же значении переносимого потока. Неконсервативное уравнение сохраняет значение вдоль характеристики, поэтому такой концентрации вещества в медленной части оно не воспроизводит.

В данном примере точные сигналы на выходе совпадают, так как входная и выходная скорости равны. Отличие моделей проявляется в профилях и в балансе: только для консервативного уравнения изменение количества внутри канала равно притоку минус отток без дополнительного члена.